## Prepare Env

In [1]:
!python -m venv venv
!source venv/bin/activate

zsh:1: command not found: python


In [2]:
%pip install boto3 pyspark delta-spark python-dotenv


[notice] A new release of pip is available: 23.2.1 -> 23.3.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os
from dotenv import load_dotenv

In [4]:
load_dotenv()

False

In [5]:
# Define S3 storage
obj_storage_access_key = os.getenv('OBJ_STORAGE_ACCESS_KEY', 'demo-access-key')
obj_storage_secret_key = os.getenv('OBJ_STORAGE_SECRET_KEY', 'demo-secret-key')
obj_storage_endpoint = os.getenv('OBJ_STORAGE_ENDPOINT', 'http://localhost:9000')

## Ingestion
### 1. Layer files to layer bronze
Write files which are in layer files to delta table in layer bronze

In [22]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("CsvToDelta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint) \
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key) \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

In [49]:
file_path = "s3a://warehouse/files/moldova.file/data.csv"
delta_table_path = "s3a://warehouse/bronze/moldova_entities.delta"

In [54]:
# Read file into a DataFrame
df = spark.read.csv(file_path, header=True)


In [55]:
df.show()

+----------------+------------------+--------------------+--------------------+--------------------+------------------------------------------------+---------------------------------------------+-------------------------------------------------------------------+---------------------------------+-------------------------------+---------------+
|IDNO/ Cod fiscal|Data înregistrării|  Denumirea completă|   Forma org./jurid.|              Adresa|Codul unităţii administrativ-teritoriale (CUATM)|Lista conducătorilor (cu indicarea rolurilor)|Lista fondatorilor (cu indicarea cotei părţi în capitalul social %)|Genuri de activitate nelicentiate|Genuri de activitate licentiate|Data lichidării|
+----------------+------------------+--------------------+--------------------+--------------------+------------------------------------------------+---------------------------------------------+-------------------------------------------------------------------+---------------------------------+-----------

In [56]:
# Write DataFrame to Delta table
df.write.format("delta").mode("overwrite").save(delta_table_path)

# Stop the Spark session
spark.stop()

AnalysisException: [DELTA_INVALID_CHARACTERS_IN_COLUMN_NAMES] Found invalid character(s) among ' ,;{}()\n\t=' in the column names of your schema. 
Please enable Column Mapping on your Delta table with mapping mode 'name'.
You can use one of the following commands.

If your table is already on the required protocol version:
ALTER TABLE table_name SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')

If your table is not on the required protocol version and requires a protocol upgrade:
ALTER TABLE table_name SET TBLPROPERTIES (
   'delta.columnMapping.mode' = 'name',
   'delta.minReaderVersion' = '2',
   'delta.minWriterVersion' = '5')


# Processing

Processing these step before writing data to layer silver
1. Transform to standard schema of layer silver
2. Unique each record
3. Add fields
4. Map entities
5. Upsert

Read delta table and discovery data

In [57]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("CsvToDelta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint) \
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key) \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

In [36]:
delta_table_path = "s3a://warehouse/bronze/moldova_entities.delta"

In [37]:
# Read Delta table
df = spark.read.format("delta").load(delta_table_path)

In [38]:
df.show()

+-----------+------------------+--------------------+--------------------+--------------------+-----------------+--------------+-------+--------------------+----+--------------------+----------+----------+------+--------------------+-----+---------+---------+---------+-------+-------------------+
|    regcode|              sepa|                name|  name_before_quotes|      name_in_quotes|name_after_quotes|without_quotes|regtype|        regtype_text|type|           type_text|registered|terminated|closed|             address|index|addressid|   region|     city|   atvk|reregistration_term|
+-----------+------------------+--------------------+--------------------+--------------------+-----------------+--------------+-------+--------------------+----+--------------------+----------+----------+------+--------------------+-----+---------+---------+---------+-------+-------------------+
|40103922165|LV34ZZZ40103922165|"Sabiedrība ar ie...|Sabiedrība ar ier...|             Regalia|           